
## Renewable - Model 1

#### An Example of optimization modeling using PuLP

---

## Mathematical Formulation

### Parameters
*   $c$: per month unit inventory cost of panels 
*   $inv0$: finished good inventory at the start of planning period
*   $q_{m}$: available supply of frames in month $m \in M$
*   $p_{m}$: price of frame in month $m \in M$
*   $d_{m}$: demand of panels in month $m \in M$ 

### Decision variables
*   $x_{m}$: number of panels produced in month $m \in M$; same as number of frames ordered in month $m \in M$ (assumed continuous variable)
*   $i_{m}$: finished goods (panel) inventory at the end of month $m \in M$ (assumed continuous variable)

### Objective function

\begin{aligned}
\min_{x,inv} \quad & \sum\limits_{m \in M} (p_{m} \; x_{m} + c \;  i_{m}) & & & 
\end{aligned}

### Constraints

\begin{aligned}
\textrm{(supply)} \quad & x_{m} & \leq & \quad q_{m}, & \forall m \in M, \\
\end{aligned}


\begin{aligned}
\textrm{(inventory, m=1)} \quad & i_{1} & = & \quad inv0 + x_{1} - d_{1}, &  \\
\end{aligned}


\begin{aligned}
\textrm{(inventory, m >= 2)} \quad & i_{m} & = & \quad i_{m-1} + x_{m} - d_{m}, & \forall m \in M, \\
\end{aligned}

\begin{aligned}
\textrm{(Nonnegativity of inventory)} \quad & i_{m} & \geq & \quad 0, & \forall m \in M, \\
\end{aligned}

\begin{aligned}
\textrm{(Nonnegativity of finished goods)} \quad & x_{m} & \geq & \quad 0, & \forall m \in M. \\
\end{aligned}

## Optimization using PuLP

### Step 1: Setup

#### Import required packages

In [1]:
import numpy as np
import pandas as pd
from pulp import *

#### Define or read-in problem parameters and data

In [2]:
c = 2000
inv0 = 0

In [3]:
d_Jan = 200
d_Feb = 300
d_Mar = 300
d_Apr = 600

In [4]:
q_Jan = 500
q_Feb = 500
q_Mar = 300
q_Apr = 300

In [5]:
p_Jan = 3500
p_Feb = 3400
p_Mar = 3800
p_Apr = 5100

### Step 2: Create Model Object

In [6]:
renewable_LP = LpProblem("Renewable",LpMinimize)

### Step 3: Add Decision Variables

In [7]:
x_Jan = LpVariable('Jan prod', lowBound=0, cat='Continuous')
x_Feb = LpVariable('Feb prod', lowBound=0, cat='Continuous')
x_Mar = LpVariable('Mar prod', lowBound=0, cat='Continuous')
x_Apr = LpVariable('Apr prod', lowBound=0, cat='Continuous')

In [8]:
i_Jan = LpVariable('Jan inv', lowBound=0, cat='Continuous')
i_Feb = LpVariable('Feb inv', lowBound=0, cat='Continuous')
i_Mar = LpVariable('Mar inv', lowBound=0, cat='Continuous')
i_Apr = LpVariable('Apr inv', lowBound=0, cat='Continuous')

### Step 4: Add Objective Function and Constraints

#### Objective Function

In [9]:
supply_cost = p_Jan * x_Jan + p_Feb * x_Feb + p_Mar * x_Mar + p_Apr * x_Apr
inventory_cost = c * (i_Jan + i_Feb + i_Mar + i_Apr)

In [10]:
renewable_LP += supply_cost + inventory_cost, 'total_cost'

#### Constraints

In [11]:
# Supply
renewable_LP += x_Jan <= q_Jan, "supply_Jan"
renewable_LP += x_Feb <= q_Feb, "supply_Feb"
renewable_LP += x_Mar <= q_Mar, "supply_Mar"
renewable_LP += x_Apr <= q_Apr, "supply_Apr"

In [12]:
# End-of-Jan inventory 
renewable_LP += i_Jan == inv0 + x_Jan - d_Jan, "inventory_Jan"

In [13]:
# End-of-month inventory, Feb - Apr
renewable_LP += i_Feb == i_Jan + x_Feb - d_Feb, "inventory_Feb"
renewable_LP += i_Mar == i_Feb + x_Mar - d_Mar, "inventory_Mar"
renewable_LP += i_Apr == i_Mar + x_Apr - d_Apr, "inventory_Apr"

#### Display / Save Formulation (Optional) 

In [14]:
renewable_LP
#renewable_LP.writeLP("renewable_LP.lp") #optional

Renewable:
MINIMIZE
2000*Apr_inv + 5100*Apr_prod + 2000*Feb_inv + 3400*Feb_prod + 2000*Jan_inv + 3500*Jan_prod + 2000*Mar_inv + 3800*Mar_prod + 0
SUBJECT TO
supply_Jan: Jan_prod <= 500

supply_Feb: Feb_prod <= 500

supply_Mar: Mar_prod <= 300

supply_Apr: Apr_prod <= 300

inventory_Jan: Jan_inv - Jan_prod = -200

inventory_Feb: Feb_inv - Feb_prod - Jan_inv = -300

inventory_Mar: - Feb_inv + Mar_inv - Mar_prod = -300

inventory_Apr: Apr_inv - Apr_prod - Mar_inv = -600

VARIABLES
Apr_inv Continuous
Apr_prod Continuous
Feb_inv Continuous
Feb_prod Continuous
Jan_inv Continuous
Jan_prod Continuous
Mar_inv Continuous
Mar_prod Continuous

### Step 5: Run solver

In [15]:
renewable_LP.solve()
print("Status:", LpStatus[renewable_LP.status])

Status: Optimal


### Step 6: Format PuLP Solution Output

In [16]:
print(f"Total cost = {value(renewable_LP.objective):,.0f}")

Total cost = 6,820,000


In [17]:
print(f"Production for the month of Jan is {x_Jan.varValue}")  
print(f"Production for the month of Feb is {x_Feb.varValue}")  
print(f"Production for the month of Mar is {x_Mar.varValue}")  
print(f"Production for the month of Apr is {x_Apr.varValue}")  

Production for the month of Jan is 300.0
Production for the month of Feb is 500.0
Production for the month of Mar is 300.0
Production for the month of Apr is 300.0


In [18]:
print(f"Inventory at end of month of Jan is {i_Jan.varValue}")  
print(f"Inventory at end of month of Feb is {i_Feb.varValue}")  
print(f"Inventory at end of month of Mar is {i_Mar.varValue}")  
print(f"Inventory at end of month of Apr is {i_Apr.varValue}")  

Inventory at end of month of Jan is 100.0
Inventory at end of month of Feb is 300.0
Inventory at end of month of Mar is 300.0
Inventory at end of month of Apr is 0.0


In [19]:
print(renewable_LP.constraints['supply_Mar'].pi)

-3700.0


---
**END**